# Data Analysis & Modeling: Classify a Content Submission into Three Categories

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
This follows the same format as [Will a Player Return?](codesignal_roblox_retention_practice.ipynb): **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

Before a content submission is reviewed, predict one coarse category: **clean**, **spam**, or **abuse**. Treat this as a mutually exclusive three-class teaching task, not Roblox's actual moderation policy.

**Problem 1 (14 minutes):** inspect and clean two supplied tables, join metadata, and summarize class balance by content type.
**Problem 2 (21 minutes):** train multinomial logistic regression, inspect the three probabilities, evaluate per-class errors, and score a new submission.

This mock practices tabular ML. It does not ask you to train a text encoder. Provided numeric signals stand in for features computed at upload time.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Compared with Will a Player Return?** Keep the familiar cleaning, merge, stratified split and sklearn Pipeline. The main changes are **three labels instead of two**, **N-by-3 probabilities**, **argmax instead of a 0.5 threshold**, and **multiclass metrics**.

**Practice version.** Replace each NotImplementedError cell with your code, then run its supplied check cell. Imports, inputs and checks are provided. Run All intentionally stops at unfinished exercises. Use the matching solution notebook only when reviewing.

## Data contract — supplied by the interviewer

The interviewer supplies **submissions.csv** (one row per submission, plus repeated export rows) and **creators.csv** (unique creator metadata). Data are synthetic. One unique creator per submission is deliberate, making a simple stratified split valid for this mock; real repeated creators may require a group split.

| Column | Meaning / policy |
| --- | --- |
| submission_id, creator_id | Join/identity columns, not numeric features |
| text_length | Number of characters at upload; negative sentinel means unknown |
| link_count | Number of links at upload |
| uppercase_ratio | Fraction of letters in uppercase, in [0,1] |
| repeated_token_ratio | Fraction of repeated tokens, in [0,1] |
| content_type | chat, description, or comment; may be missing |
| creators.account_age_days | Account age at upload; may be missing |
| label | Reviewer-provided class, available only after review |
| review_minutes | Time spent on the subsequent review; **leakage**, never a feature |

Every submission has a known label and matching creator. Drop exact duplicates; conflicting submission IDs should fail a check. Keep rows with incomplete predictors.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox_softmax/submissions.csv** and **creators.csv**. The provided cell imports libraries and sets DATA_DIR. Load both files with pd.read_csv in Step 1; do not modify the source CSVs.

Start the 35-minute coding timer after setup.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, precision_score, recall_score)
SEED = 42

DATA_DIR = Path("data/codesignal_roblox_softmax")
assert (DATA_DIR / "submissions.csv").exists(), "Run from the workspace root."
assert (DATA_DIR / "creators.csv").exists()


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Inspect submissions and creators. Count duplicate exports, invalid text lengths and missing fields. Inspect the label counts, but do not treat arbitrary category codes as ordered targets.

**Say before coding:** "This is single-label multiclass classification: each example belongs to exactly one of three classes."

**Possible interviewer questions**

1. Why not regression with clean=0, spam=1, abuse=2?
2. Could content actually be both spam and abuse?
3. What should you inspect before dropping missing rows?


In [20]:
# Create the variables named in the task above.
#
DATA_DIR = "data/codesignal_roblox_softmax/"
submission_path = DATA_DIR + "submissions.csv"
creator_path = DATA_DIR + "creators.csv"


submissions = pd.read_csv(submission_path)

creators = pd.read_csv(creator_path)


#print("duplicates:", submissions.duplicated())

mask = submissions.duplicated()
display(submissions.loc[mask])

#print("duplicates:", creators.duplicated().sum())

print("invalidate text length", submissions["text_length"].lt(0).sum())
print("missing filed", submissions.isna().sum())

print("label counts", submissions["label"].value_counts(normalize=True))
#display(submissions.info())
#display(submissions.head())
#display(submissions.describe())

#display(creators.info())
#display(creators.describe())
#display(creators.head())
# raise NotImplementedError("Complete step 1, then run the checks below.")


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes
1200,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918
1201,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087
1202,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863
1203,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839
1204,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389
1205,5,5,559.0,2,0.196848,0.003399,comment,clean,1.234961
1206,6,6,65.0,2,0.379931,0.409406,chat,clean,1.133420
1207,7,7,456.0,0,0.056156,0.030394,chat,clean,1.055178
1208,8,8,138.0,0,0.356277,0.372899,description,clean,1.976692
1209,9,9,70.0,1,0.094415,0.103063,description,clean,1.536777


invalidate text length 35
missing filed submission_id            0
creator_id               0
text_length              0
link_count               0
uppercase_ratio          0
repeated_token_ratio     0
content_type            40
label                    0
review_minutes           0
dtype: int64
label counts label
clean    0.586831
abuse    0.235391
spam     0.177778
Name: proportion, dtype: float64


In [ ]:
assert creators["creator_id"].is_unique
assert submissions["creator_id"].isin(creators["creator_id"]).all()
assert set(submissions["label"]) == {"clean", "spam", "abuse"}


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create clean by dropping exact duplicates and replacing negative text_length with NaN. Left merge creators into frame with validate="many_to_one". Do not use review_minutes as a feature.

**Say before coding:** "The creator table has one row per key. Merge validation prevents accidental row multiplication."

**Possible interviewer questions**

1. Why many_to_one?
2. Why is review_minutes leakage?
3. Why postpone median imputation until after splitting?


In [28]:
# Create the variables named in the task above.
cleaned_submissions = submissions.drop_duplicates()

#print("duplicates:", cleaned_submissions.duplicated().sum())

mask = cleaned_submissions["text_length"].lt(0)

cleaned_submissions.loc[mask, "text_length"] = np.nan

#display(cleaned_submissions.loc[mask])

frame = cleaned_submissions.merge(creators, on='creator_id', how="left", validate="many_to_one" )

display(frame)
#raise NotImplementedError("Complete step 2, then run the checks below.")


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes,account_age_days
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918,1395.0
1,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087,614.0
2,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863,1387.0
3,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839,124.0
4,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389,480.0
...,...,...,...,...,...,...,...,...,...,...
1195,1195,1195,415.0,2,0.209577,0.281403,description,clean,1.229651,596.0
1196,1196,1196,466.0,1,0.287657,0.114925,chat,abuse,5.116780,345.0
1197,1197,1197,638.0,1,0.488615,0.230904,chat,abuse,5.377732,988.0
1198,1198,1198,523.0,0,0.155566,0.256894,comment,clean,1.486195,427.0


In [ ]:
assert len(frame) == submissions["submission_id"].nunique()
assert frame["text_length"].dropna().ge(0).all()
assert frame["submission_id"].is_unique


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Create class_fraction from the cleaned frame. Display counts by content_type and label, including missing content types. Identify the largest class and say why accuracy alone may hide poor minority recall.

**Say before coding:** "I will check each class instead of relying only on the overall score."

**Possible interviewer questions**

1. What does normalize=True change?
2. Should we immediately balance the dataset?
3. How do you check that a merge did not multiply rows?


In [32]:
# Create the variables named in the task above.
class_fraction = frame["label"].value_counts()

print("class_fraction",class_fraction)

print("counts by content_type", frame["content_type"].value_counts()
)

display(frame.groupby(
    ["content_type", "label"],
    dropna=False
).size())

#raise NotImplementedError("Complete step 3, then run the checks below.")


class_fraction label
clean    701
abuse    285
spam     214
Name: count, dtype: int64
counts by content_type content_type
comment        411
description    382
chat           367
Name: count, dtype: int64


content_type  label
chat          abuse     94
              clean    204
              spam      69
comment       abuse    107
              clean    243
              spam      61
description   abuse     78
              clean    226
              spam      78
NaN           abuse      6
              clean     28
              spam       6
dtype: int64

In [ ]:
np.testing.assert_allclose(class_fraction.sum(), 1)
assert len(class_fraction) == 3


## Problem 2 · Step 4 — Select features and split (14–18 min)

Use text_length, link_count, uppercase_ratio, repeated_token_ratio, account_age_days as numeric_features and content_type as categorical_features. Create an 80/20 stratified split with SEED.

**Say before coding:** "Stratification approximately preserves the three-class proportions; preprocessing still fits only on train."

**Possible interviewer questions**

1. Does stratification make the classes equally common?
2. What if creators appear in both splits in production data?
3. How do you check that X and y still correspond?


In [45]:
# Create the variables named in the task above.
numeric_features = ["text_length", "link_count", "uppercase_ratio", "repeated_token_ratio", "account_age_days"]
categorical_features = ["content_type"]

X = frame[numeric_features + categorical_features]
y = frame["label"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)



#raise NotImplementedError("Complete step 4, then run the checks below.")


In [43]:
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
assert not {"label", "review_minutes", "creator_id", "submission_id"} & set(X.columns)
assert y_train.nunique() == y_test.nunique() == 3


AssertionError: 

## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Use the familiar numeric and categorical pipelines. Create model with LogisticRegression(solver="lbfgs", max_iter=500), and baseline with DummyClassifier(strategy="prior"). Fit both on train only.

**Say before coding:** "With three classes, this solver fits multinomial logistic regression: one logit per class, followed by softmax."

**Possible interviewer questions**

1. What is the model mathematically?
2. Does each class probability use an independent sigmoid?
3. What happens when you call model.fit on the Pipeline?


In [47]:
# Create the variables named in the task above.
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("normalize", StandardScaler()),
])


categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression()),
])


print("y_train",y_train)
model.fit(X_train,y_train)



help(SimpleImputer)



#raise NotImplementedError("Complete step 5, then run the checks below.")


y_train 331     clean
409     clean
76      clean
868     clean
138     clean
        ...  
1044     spam
1095    clean
1130    clean
860     clean
1126    clean
Name: label, Length: 960, dtype: object
Help on class SimpleImputer in module sklearn.impute._base:

class SimpleImputer(_BaseImputer)
 |  SimpleImputer(*, missing_values=nan, strategy='mean', fill_value=None, copy=True, add_indicator=False, keep_empty_features=False)
 |  
 |  Univariate imputer for completing missing values with simple strategies.
 |  
 |  Replace missing values using a descriptive statistic (e.g. mean, median, or
 |  most frequent) along each column, or using a constant value.
 |  
 |  Read more in the :ref:`User Guide <impute>`.
 |  
 |  .. versionadded:: 0.20
 |     `SimpleImputer` replaces the previous `sklearn.preprocessing.Imputer`
 |     estimator which is now removed.
 |  
 |  Parameters
 |  ----------
 |  missing_values : int, float, str, np.nan, None or pandas.NA, default=np.nan
 |      The placehol

In [ ]:
assert model.named_steps["classifier"].coef_.shape[0] == 3
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create prob and pred. Display the first five probability rows with model.classes_ as column names. Evaluate model and baseline with accuracy, macro-F1 and log loss. Display classification_report and a labeled 3x3 confusion matrix.

**Say before coding:** "Argmax selects one class. Macro-F1 weights classes equally; log loss also evaluates confidence."

**Possible interviewer questions**

1. Why not threshold every probability at 0.5?
2. Why does log loss need probabilities?
3. Can you keep changing the model after looking at test metrics?


In [53]:
# Create the variables named in the task above.
pred = model.predict(X_test)
prob = model.predict_proba(X_test)
display(pd.DataFrame(prob[:5], columns=model.classes_))
display(prob)
display(pred.shape)
print(X_test.shape)
def evaluation(gt, prediction):
    return {
        "accuracy": accuracy_score(gt,prediction),
        "macro_f1": f1_score(gt,prediction, average="macro", zero_division=0),
    }

scores = pd.DataFrame({"softmax": evaluation(y_test,pred)})

display(scores)


#raise NotImplementedError("Complete step 6, then run the checks below.")


,abuse,clean,spam
0,0.651465,0.340499,0.008036
1,0.108269,0.549979,0.341752
2,0.178568,0.800486,0.020947
3,0.267928,0.599335,0.132737
4,0.349483,0.594289,0.056228


array([[0.65146521, 0.34049884, 0.00803595],
       [0.10826861, 0.54997895, 0.34175244],
       [0.17856764, 0.80048581, 0.02094655],
       [0.26792798, 0.59933473, 0.13273729],
       [0.34948285, 0.59428903, 0.05622812],
       [0.11143794, 0.74235141, 0.14621065],
       [0.06404754, 0.35645481, 0.57949764],
       [0.08629989, 0.35210679, 0.56159332],
       [0.06145122, 0.82678236, 0.11176642],
       [0.15208014, 0.78966516, 0.0582547 ],
       [0.42106805, 0.39350088, 0.18543106],
       [0.22978812, 0.65319209, 0.11701979],
       [0.49521309, 0.46522951, 0.0395574 ],
       [0.22897592, 0.73323012, 0.03779396],
       [0.22124182, 0.58455033, 0.19420785],
       [0.66210629, 0.32641862, 0.01147509],
       [0.2183408 , 0.74564692, 0.03601228],
       [0.0868245 , 0.57031072, 0.34286478],
       [0.16798819, 0.81473406, 0.01727775],
       [0.17806352, 0.76040359, 0.0615329 ],
       [0.67256816, 0.2710603 , 0.05637154],
       [0.05304326, 0.52814133, 0.41881542],
       [0.

(240,)

(240, 6)


,softmax
accuracy,0.637500
macro_f1,0.527465


In [ ]:
assert prob.shape == (len(y_test), 3)
np.testing.assert_allclose(prob.sum(axis=1), 1)
np.testing.assert_array_equal(pred, model.classes_[prob.argmax(axis=1)])
true_col = pd.Index(model.classes_).get_indexer(y_test)
np.testing.assert_allclose(scores.loc["softmax", "log_loss"],
    -np.log(prob[np.arange(len(y_test)), true_col]).mean())


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create new_submission with a missing account age and unseen content_type="profile_bio". Produce new_prob and new_label. Explain how you would handle uncertain predictions without treating this toy model as a production moderation system.

**Say before coding:** "The model returns a category and a confidence distribution; deployment needs a separate reviewed policy."

**Possible interviewer questions**

1. Does a probability of 0.8 guarantee 80% correctness?
2. How might we use a human review queue?
3. What must training and inference share?


In [ ]:
# Create the variables named in the task above.
raise NotImplementedError("Complete step 7, then run the checks below.")


In [ ]:
assert new_prob.shape == (1, 3)
np.testing.assert_allclose(new_prob.sum(axis=1), 1)


## If the interviewer extends the problem

**Optional 5-minute mathematical follow-up:** reconstruct predict_proba from logits. In the reference, the next cell verifies it numerically. This is not required in the 35-minute exercise.

For one row: subtract its maximum logit, exponentiate, then divide by the row sum. Subtracting a constant leaves softmax unchanged and avoids overflow.


## Scope and sources

These are **original synthetic exercises**, not reported Roblox questions and not official CodeSignal assessments. The user-supplied **Data Analysis & Modeling Coding Interview** guide defines the format: 5 minutes intro, 35 minutes for two problems (analysis and modeling), 5 minutes Q&A; pandas, NumPy, sklearn and SciPy are available. The per-step times here are suggested practice checkpoints, not official grading criteria.

The guide does not identify a team or prescribe regression/softmax topics. We chose those to practice task-appropriate modeling and metrics. Business themes are informed by the [official PhD Early Career MLE posting](https://careers.roblox.com/jobs/8242623), which mentions engagement, safety and recommendation systems. Source checked 2026-10-22; exact interview questions are unknown.

Implementation references: [sklearn 1.6 LogisticRegression](https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html), [model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
